# Stellantis_regional_sales: Outlier analysis and SPC
**Author:** [Joel Perez Guerrero]  
**Tools:** Python, pgAdmin, Pandas.  
-En este notebook realizamos la extracción de datos desde PostgreSQL y aplicamos técnicas de detección de anomalías.

1. Upload DB from postgresql

In [13]:
# DATA BASE CONNECTION SETTINGS
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv('../.env')

def get_connection():
    try:
        # 2. Get credentials
        user = os.getenv('DB_USER')
        password = os.getenv('DB_PASS') 
        host = os.getenv('DB_HOST')
        port = os.getenv('DB_PORT')
        db = os.getenv('DB_NAME')

        missing_vars = [var for var, val in {
            'DB_USER': user, 'DB_PASS': password, 
            'DB_HOST': host, 'DB_PORT': port, 'DB_NAME': db
        }.items() if val is None]

        if missing_vars:
            raise ValueError(f"Faltan variables en el .env: {', '.join(missing_vars)}")

        #Create connection engine
        url_conexion = f'postgresql://{user}:{password}@{host}:{port}/{db}'
        engine = create_engine(url_conexion)
        
        # Conection test
        with engine.connect() as conn:
            print(f"✅ Conexión establecida con éxito a la base de datos: {db}")
        
        return engine

    except Exception as e:
        print(f"❌ Error al conectar: {e}")
        return None

# Execute connection
engine = get_connection()

#if engine is None:
    # Esto detendrá la ejecución del notebook si falla la conexión
    #raise Exception("No se puede continuar sin conexión a la base de datos.")

✅ Conexión establecida con éxito a la base de datos: stellantis_db


2. We run query to join tables and define dataframe.
- We need to clasify outliers using sales amount and brands. So fact_ventas and dim_vehiculos are joined.
- Create "raw_data" file

In [14]:
#FROM DB LOAD DATA AS DATAFRAME

query = """

SELECT *
FROM fact_ventas_cl v
JOIN dim_vehiculos_cl u
ON v.id_vehiculo = u.id_vehiculo
ORDER BY v.id_transaccion
;

"""
if engine:
    # Read raw data using query
    df = pd.read_sql(query, engine)
    df= df.loc[:, ~df.columns.duplicated()] #Drop duplicated columns if exist
    df.to_parquet('../data/raw_data.parquet', index=False)
    print(f"📊 Dataset cargado: {df.shape[0]} filas extraídas.")
    display(df.head(10))
else:
    print("❌ No se pudo cargar data frame de la base de datos.")


📊 Dataset cargado: 536321 filas extraídas.


,id_transaccion,id_vehiculo,id_ubicacion,id_cliente,fecha,unidades,venta_bruta_sn,modelo,marcas_cl
0,ST-2000000,1,1,4446,2024-11-04,1,347951.0,5008,Peugeot
1,ST-2000001,2,2,5967,2025-10-20,1,808947.0,1500,RAM
2,ST-2000002,3,3,1084,2023-12-24,1,533672.0,Pulse,Fiat
3,ST-2000003,4,4,12388,2023-07-27,1,493390.0,Grand Cherokee,Jeep
4,ST-2000004,5,2,2022,2025-10-25,2,1698334.0,ProMaster,RAM
5,ST-2000005,6,5,2466,2024-01-25,1,902230.0,Ducato,Fiat
6,ST-2000006,7,5,2355,2023-01-09,1,927566.0,ProMaster,RAM
7,ST-2000007,8,6,14630,2025-09-07,1,359892.0,Argo,Fiat
8,ST-2000008,5,2,1228,2023-04-21,1,570192.0,ProMaster,RAM
9,ST-2000009,5,3,1316,2022-05-16,1,826663.0,ProMaster,RAM
